Transactions

      ↓

Text Cleaning

      ↓

ML Discovery (K-Means)

      ↓

Underwriting Rules

      ↓

Customer Risk Profile

      ↓      
Loan Decision Support

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

#data loaded
df = pd.read_csv('Sample AA Data.csv')
# Drop rows with no narration
df = df.dropna(subset=['NARRATION'])

# Clean AMOUNT
if 'AMOUNT' in df.columns:
    df['AMOUNT'] = df['AMOUNT'].astype(str).str.replace(',', '', regex=False)
    df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce').fillna(0)

# Clean TYPE
if 'TYPE' in df.columns:
    df['TYPE'] = df['TYPE'].astype(str).str.strip().str.upper()

def clean_text(text):
    if pd.isna(text): return ""
                # * pd.isna() is a Pandas function that checks whether a value is missing (NaN, None, etc.).* If text is missing, the function returns an empty string
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['CLEAN_NARRATION'] = df['NARRATION'].apply(clean_text)

In [ ]:
#Machine Learning
#Clean narration → Convert words to numerical vectors (TF-IDF) → Group similar transactions using K-Means → Store the cluster number in ML_CLUSTER_TAG.

vectorizer = TfidfVectorizer(stop_words='english', max_features=300)
X = vectorizer.fit_transform(df['CLEAN_NARRATION'])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['ML_CLUSTER_TAG'] = kmeans.fit_predict(X)

In [ ]:
df['UPPER_NARRATION'] = df['CLEAN_NARRATION'].str.upper()

In [ ]:
#1. CHANNEL Determines how money moved. upi digital bank transfer
def get_payment_channel(text):
    if any(w in text for w in ['UPI', 'PHONEPE', 'GPAY']): return 'UPI/Digital'
    if any(w in text for w in ['IMPS', 'NEFT', 'RTGS']): return 'Bank Transfer'
    if 'ATM' in text or 'CASH' in text: return 'Cash'
    if 'NACH' in text or 'ACH' in text: return 'Auto-Debit'
    return 'Other'
df['M1_CHANNEL'] = df['UPPER_NARRATION'].apply(get_payment_channel)

In [ ]:
# 2. CASH FLOW
def get_cash_flow(row):
    text = row['UPPER_NARRATION']
    txn_type = str(row.get('TYPE', ''))
    if txn_type == 'CREDIT':
        if 'SALARY' in text or 'PAYROLL' in text: return 'Core Income (Salary)'
        return 'General Inflow'
    else:
        if any(w in text for w in ['ZOMATO', 'SWIGGY', 'AMAZON', 'FLIPKART']): return 'Discretionary Spend'
        if any(w in text for w in ['RELIANCE', 'MART', 'GROCERY']): return 'Essential Spend'
        return 'General Outflow'
df['M2_CASH_FLOW'] = df.apply(get_cash_flow, axis=1)

In [ ]:
# 3. OBLIGATION(debt)
def get_obligations(text):
    if any(w in text for w in ['EMI', 'LOAN', 'FINANCE', 'BAJAJ']): return 'Active EMI / Debt'
    if 'CREDIT CARD' in text or 'SBI CARD' in text: return 'Credit Card Bill'
    return 'No Debt Detected'
df['M3_OBLIGATION'] = df['UPPER_NARRATION'].apply(get_obligations)

In [ ]:
# 4. RISK
def get_risk_flags(text):
    if any(w in text for w in ['BOUNCE', 'RTN', 'INSUFFICIENT', 'PENALTY']): return 'HIGH RISK: Penalty/Bounce'
    if any(w in text for w in ['DREAM', 'RUMMY', 'CASINO', 'CRYPTO']): return 'HIGH RISK: Gambling/Crypto'
    return 'Clean'
df['M4_RISK_FLAG'] = df['UPPER_NARRATION'].apply(get_risk_flags)

In [ ]:
# 5. WEALTH
def get_wealth_flags(text):
    if any(w in text for w in ['SIP', 'MUTUAL FUND', 'ZERODHA', 'GROWW']): return 'Investment (Equity/MF)'
    if any(w in text for w in ['LIC', 'INSURANCE', 'MEDICLAIM']): return 'Protected (Insurance)'
    if 'FD' in text or 'RD' in text: return 'Savings (FD/RD)'
    return 'None'
df['M5_WEALTH'] = df['UPPER_NARRATION'].apply(get_wealth_flags)

In [ ]:
# 6. BUSINESS
def get_business_flags(text):
    if any(w in text for w in ['DISTRIBUTOR', 'TRADER', 'WHOLESALE', 'GST']): return 'Business/Commercial Activity'
    return 'Personal'
df['M6_BUSINESS_MIX'] = df['UPPER_NARRATION'].apply(get_business_flags)

In [ ]:
# 7. GEOGRAPHY
def extract_geography(text):
    # Search the RAW narration for POS geography since cleaning removes numbers
    raw_text = str(text).upper()
    if 'POS' in raw_text:
        match = re.search(r'POS.*?([A-Z]+)$', raw_text)
        if match: return match.group(1)
    return 'Unknown'
df['M7_GEOGRAPHY'] = df['NARRATION'].apply(extract_geography)

In [ ]:
output_file = 'nlP_Classification_Analyzed_Data.csv'
df.to_csv(output_file, index=False)

print("SUMMARY")


total_txns = len(df)
risk_txns = len(df[df['M4_RISK_FLAG'] != 'Clean'])
emi_txns = len(df[df['M3_OBLIGATION'] != 'No Debt Detected'])
wealth_txns = len(df[df['M5_WEALTH'] != 'None'])

# Calculate percentages for a more professional report
risk_pct = (risk_txns / total_txns) * 100 if total_txns > 0 else 0

print("PORTFOLIO METRICS:")
print(f"- Total Valid Transactions Analyzed : {total_txns}")
print(f"- Identified Active Debt/EMI Hits   : {emi_txns}")
print(f"- Identified Wealth & Savings Hits  : {wealth_txns}")
print(f"- Detected Risk/Penalty Flags       : {risk_txns} ({risk_pct:.1f}% of portfolio)")


print("EXECUTIVE ASSESSMENT & RECOMMENDATION:")

if total_txns > 0 and risk_txns > (total_txns * 0.05):
    print("Risk Assessment: Elevated Risk Profile")

elif wealth_txns > 0 and risk_txns == 0:
    print("Risk Assessment: Prime Profile")

else:
    print("Risk Assessment: Standard Profile")

print(f"Data successfully exported to: {output_file}")

SUMMARY
PORTFOLIO METRICS:
- Total Valid Transactions Analyzed : 87
- Identified Active Debt/EMI Hits   : 0
- Identified Wealth & Savings Hits  : 0
- Detected Risk/Penalty Flags       : 0 (0.0% of portfolio)
EXECUTIVE ASSESSMENT & RECOMMENDATION:
Risk Assessment: Standard Profile
Data successfully exported to: nlP_Classification_Analyzed_Data.csv
